# Complete Inference Pipeline

End-to-end pipeline: Image → Embedding → Abnormality → Classification → Retrieval

In [1]:
import torch
import open_clip
from PIL import Image
import json
from pathlib import Path

c:\Users\Harsh\Desktop\major-project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Import Components

Copy the classes from previous notebooks or import them if you've converted to .py files

In [6]:
# Run the cells from previous notebooks to define classes
# Or import if you have .py modules

%run 01_embeddings.ipynb
%run 02_classification.ipynb
%run 03_abnormality.ipynb
%run 04_retrieval.ipynb

Loading hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224...
Model loaded on cuda


C:\Users\Harsh\AppData\Local\Temp\ipykernel_59684\665958857.py:3: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\autograd\python_variable_indexing.cpp:351.)
  embeddings = data['embeddings']


IndexError: too many indices for tensor of dimension 2

IndexError: too many indices for tensor of dimension 2

## Medical Imaging Pipeline Class

In [3]:
class MedicalImagingPipeline:
    def __init__(self):
        print("🔧 Initializing Medical Imaging Pipeline...")
        
        # Load all components
        print("  Loading embedder...")
        self.embedder = BiomedCLIPEmbedder()
        
        print("  Loading classifier...")
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        self.classifier = load_classifier(device=self.device)
        
        print("  Loading abnormality detector...")
        self.abnormality_detector = AbnormalityDetector()
        
        print("  Loading retriever...")
        self.retriever = MedicalImageRetriever()
        
        # CheXpert labels
        self.labels = [
            'Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema',
            'Enlarged Cardiomediastinum', 'Fracture', 'Lung Lesion',
            'Lung Opacity', 'Pleural Effusion', 'Pleural Other',
            'Pneumonia', 'Pneumothorax', 'Support Devices', 'No Finding'
        ]
        
        print("✅ Pipeline ready!\n")
    
    def predict(self, image_path, retrieve_k=5):
        """
        Run full pipeline on an image.
        
        Args:
            image_path: Path to chest X-ray image
            retrieve_k: Number of similar cases to retrieve per category
        
        Returns:
            dict: Complete analysis results
        """
        print(f"📊 Analyzing: {image_path}")
        
        # 1. Generate embedding
        print("  [1/4] Generating embedding...")
        embedding = self.embedder.encode_image(image_path)
        
        # 2. Abnormality detection
        print("  [2/4] Detecting abnormality...")
        abnormality = self.abnormality_detector.analyze(embedding)
        
        # 3. Multi-label classification
        print("  [3/4] Classifying diseases...")
        probs = predict(self.classifier, embedding, device=self.device)
        disease_probs = {
            label: float(prob) 
            for label, prob in zip(self.labels, probs[0])
        }
        
        # 4. Retrieve similar cases
        print("  [4/4] Retrieving similar cases...")
        retrieved = self.retriever.retrieve(
            embedding, 
            k=retrieve_k, 
            stratify=True,
            abnormality_threshold=0.5
        )
        
        print("✅ Analysis complete!\n")
        
        return {
            'image_path': str(image_path),
            'abnormality_score': abnormality['abnormality_score'],
            'abnormality_label': abnormality['abnormality_label'],
            'disease_probabilities': disease_probs,
            'top_diseases': sorted(
                disease_probs.items(), 
                key=lambda x: x[1], 
                reverse=True
            )[:5],
            'retrieved_cases': retrieved
        }
    
    def print_results(self, results):
        """Pretty print results."""
        print("="*60)
        print("ANALYSIS RESULTS")
        print("="*60)
        
        print(f"\n📁 Image: {results['image_path']}")
        
        print(f"\n🔍 Abnormality Score: {results['abnormality_score']:.4f}")
        print(f"   Classification: {results['abnormality_label'].upper()}")
        
        print("\n🏥 Top 5 Predicted Diseases:")
        for i, (disease, prob) in enumerate(results['top_diseases'], 1):
            bar = '█' * int(prob * 20)
            print(f"   {i}. {disease:30s} {prob:.3f} {bar}")
        
        print(f"\n📚 Retrieved Similar Cases:")
        print(f"   Abnormal: {len(results['retrieved_cases']['abnormal'])} cases")
        print(f"   Healthy:  {len(results['retrieved_cases']['healthy'])} cases")
        
        print("\n" + "="*60)

## Initialize Pipeline

In [4]:
pipeline = MedicalImagingPipeline()

🔧 Initializing Medical Imaging Pipeline...
  Loading embedder...


NameError: name 'BiomedCLIPEmbedder' is not defined

## Run Inference on Test Image

In [ ]:
# Replace with your test image path
test_image = 'path/to/test/xray.jpg'

# Run pipeline
results = pipeline.predict(test_image, retrieve_k=5)

# Display results
pipeline.print_results(results)

## Save Results to JSON

In [ ]:
# Save to file
output_path = 'results/test_case_results.json'

with open(output_path, 'w') as f:
    json.dump(results, f, indent=2)

print(f"✅ Results saved to {output_path}")

## Batch Processing

In [ ]:
def process_batch(pipeline, image_paths, output_dir='results/batch/'):
    """Process multiple images."""
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    
    all_results = []
    
    for img_path in image_paths:
        try:
            results = pipeline.predict(img_path)
            all_results.append(results)
            
            # Save individual result
            img_name = Path(img_path).stem
            with open(f"{output_dir}/{img_name}.json", 'w') as f:
                json.dump(results, f, indent=2)
        
        except Exception as e:
            print(f"❌ Failed on {img_path}: {e}")
    
    # Save combined results
    with open(f"{output_dir}/all_results.json", 'w') as f:
        json.dump(all_results, f, indent=2)
    
    print(f"\n✅ Processed {len(all_results)} images")
    return all_results

In [ ]:
# Example batch processing
# test_images = ['image1.jpg', 'image2.jpg', 'image3.jpg']
# batch_results = process_batch(pipeline, test_images)